<a href="https://colab.research.google.com/github/Udana-Gits/My_AI_Learnings/blob/main/L6_HMM_%2B_Agent_Ontology_%2B_Multi_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 Customer Satisfaction Simulation with Ontology + HMM + Mesa

This notebook demonstrates how to combine:

- **RDF Ontology** 🧩: defines our concepts (customers, service agents, states, observations).  
- **Hidden Markov Model (HMM)** 🎲: provides probabilistic reasoning about hidden customer moods.  
- **Mesa Multi-Agent Simulation** 👥: simulates dynamic interactions between customers and service agents.  

👉 Business Use Case: **Monitor hidden customer satisfaction**. Customers act (purchase, complain, or stay silent). Service agents observe these signals and infer true hidden moods.

## 🌍 Step 1. Install Required Packages
We need three main libraries:
- `rdflib` → defines ontology (classes, properties, individuals).  
- `hmmlearn` → implements HMM for probabilistic inference.  
- `mesa` → agent-based model framework.

In [ ]:
!pip install mesa hmmlearn rdflib

## 🧩 Step 2. Define Ontology in RDF

Our ontology describes **concepts** (classes), their **relationships** (object properties), **attributes** (data properties), and **individuals** (instances).

### 📌 Classes:
- **CustomerAgent** → simulated customers
- **ServiceAgent** → simulated staff
- **CustomerState** → hidden moods = {Happy, Neutral, Unhappy}
- **Observation** → visible behaviors = {Purchase, Complaint, Silence}

### 📌 Object Properties:
- **hasState** (CustomerAgent → CustomerState)
- **canObserve** (ServiceAgent → Observation)
- **canInfer** (ServiceAgent → CustomerState)

### 📌 Data Property:
- **hasSatisfactionScore** (CustomerAgent → float)

### 📌 Individuals:
- States: Happy, Neutral, Unhappy  
- Observations: Purchase, Complaint, Silence  
- Example customers: Customer_1, Customer_2  
- Example service agent: Agent_A

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, XSD, Literal

EX = Namespace("http://example.org/ontology#")

def build_ontology():
    g = Graph()
    g.bind("ex", EX)

    # Classes
    g.add((EX.CustomerAgent, RDF.type, RDFS.Class))
    g.add((EX.ServiceAgent, RDF.type, RDFS.Class))
    g.add((EX.CustomerState, RDF.type, RDFS.Class))
    g.add((EX.Observation, RDF.type, RDFS.Class))

    # Properties
    g.add((EX.hasState, RDF.type, RDF.Property))
    g.add((EX.canObserve, RDF.type, RDF.Property))
    g.add((EX.canInfer, RDF.type, RDF.Property))
    g.add((EX.hasSatisfactionScore, RDF.type, RDF.Property))

    # States
    for state in ["Happy", "Neutral", "Unhappy"]:
        g.add((EX[state], RDF.type, EX.CustomerState))

    # Observations
    for obs in ["Purchase", "Complaint", "Silence"]:
        g.add((EX[obs], RDF.type, EX.Observation))

    # Individuals
    g.add((EX.Customer_1, RDF.type, EX.CustomerAgent))
    g.add((EX.Customer_1, EX.hasState, EX.Happy))
    g.add((EX.Customer_1, EX.hasSatisfactionScore, Literal(0.9, datatype=XSD.float)))

    g.add((EX.Customer_2, RDF.type, EX.CustomerAgent))
    g.add((EX.Customer_2, EX.hasState, EX.Unhappy))

    g.add((EX.Agent_A, RDF.type, EX.ServiceAgent))
    g.add((EX.Agent_A, EX.canObserve, EX.Complaint))
    g.add((EX.Agent_A, EX.canInfer, EX.CustomerState))

    return g

ontology_graph = build_ontology()
print(ontology_graph.serialize(format="turtle")[:500])

## 🎲 Step 3. Hidden Markov Model (HMM)

The HMM models the **hidden moods (states)** and how they:
1. Evolve over time (transition probabilities).
2. Produce observations (emission probabilities).

### Transition Matrix
- Happy → 60% Happy, 30% Neutral, 10% Unhappy
- Neutral → 20% Happy, 50% Neutral, 30% Unhappy
- Unhappy → 10% Happy, 40% Neutral, 50% Unhappy

### Emission Matrix
- Happy → 70% Purchase, 10% Complaint, 20% Silence
- Neutral → 30% Purchase, 20% Complaint, 50% Silence
- Unhappy → 10% Purchase, 60% Complaint, 30% Silence

In [ ]:
import numpy as np
from hmmlearn import hmm

states = [s.split("#")[-1] for s in ontology_graph.subjects(RDF.type, EX.CustomerState)]
observations = [o.split("#")[-1] for o in ontology_graph.subjects(RDF.type, EX.Observation)]

transition_matrix = np.array([
    [0.6, 0.3, 0.1],
    [0.2, 0.5, 0.3],
    [0.1, 0.4, 0.5]
])

emission_matrix = np.array([
    [0.7, 0.1, 0.2],
    [0.3, 0.2, 0.5],
    [0.1, 0.6, 0.3]
])

start_probs = np.array([0.5, 0.3, 0.2])

hmm_model = hmm.CategoricalHMM(n_components=len(states), n_iter=1)
hmm_model.startprob_ = start_probs
hmm_model.transmat_ = transition_matrix
hmm_model.emissionprob_ = emission_matrix

print("States:", states)
print("Observations:", observations)

# 🤖 CustomerAgent — Producing Observations

Customers have a hidden state (Happy, Neutral, Unhappy). Each step, they **emit signals** based on probabilities (Purchase, Complaint, Silence).

✔ Example: If Happy → ~70% chance of Purchase, 20% Silence, 10% Complaint.

In [ ]:
from mesa import Agent, Model
from mesa.time import RandomActivation

class CustomerAgent(Agent):
    def __init__(self, unique_id, model, true_state=None):
        super().__init__(unique_id, model)
        self.state = true_state if true_state else self.random.choice(states)
    def step(self):
        idx = states.index(self.state)
        obs = np.random.choice(observations, p=hmm_model.emissionprob_[idx])
        self.model.observations.append((self.unique_id, obs))

# 🕵️ ServiceAgent — Inferring Hidden States

Service agents read customer observations, then use HMM decoding (Viterbi) to infer the most likely hidden state.

✔ Example: Observation = "Complaint" → HMM infers most likely state = Unhappy.

In [ ]:
class ServiceAgent(Agent):
    def step(self):
        if self.model.observations:
            cust_id, obs = self.model.observations.pop(0)
            obs_idx = observations.index(obs)
            logprob, hidden_states = hmm_model.decode(np.array([[obs_idx]]), algorithm="viterbi")
            inferred_state = states[hidden_states[0]]
            print(f"[Service Agent] Customer {cust_id}: '{obs}' → Inferred state '{inferred_state}'")

# 🏗️ CustomerServiceModel — Orchestrating Simulation

The simulation model holds the schedule, observation mailbox, and creates all customers and service agents from ontology.

In [ ]:
class CustomerServiceModel(Model):
    def __init__(self, N_customers=2, N_agents=1):
        self.schedule = RandomActivation(self)
        self.observations = []

        # Add customers
        cust_inds = [c for c in ontology_graph.subjects(RDF.type, EX.CustomerAgent)]
        for i, c in enumerate(cust_inds[:N_customers]):
            st = None
            for cs in ontology_graph.objects(c, EX.hasState):
                st = cs.split("#")[-1]
            agent = CustomerAgent(i, self, st)
            self.schedule.add(agent)

        # Add service agents
        serv_inds = [s for s in ontology_graph.subjects(RDF.type, EX.ServiceAgent)]
        for j, s in enumerate(serv_inds[:N_agents]):
            sa = ServiceAgent(100+j, self)
            self.schedule.add(sa)
    def step(self):
        self.schedule.step()

## ▶️ Run Simulation

In [ ]:
sim = CustomerServiceModel(N_customers=2, N_agents=1)
for t in range(5):
    print(f"\n--- Step {t} ---")
    sim.step()

# 📊 Understanding the Output

A typical output looks like:
```
--- Step 0 ---
[Service Agent] Customer 0: 'Purchase' → Inferred state 'Happy'
[Service Agent] Customer 1: 'Complaint' → Inferred state 'Unhappy'
```

### How to read it:
- **Step number** → simulation tick number
- **Customer observation** → what a customer did (Purchase, Complaint, Silence)
- **Inferred state** → service agent’s best guess of hidden mood via HMM

🎯 This shows how individual **hidden states (ontology) → generate behavior (HMM emission) → get interpreted (ServiceAgent)**.

💡 Important: Because of randomness, runs vary! Sometimes agents guess wrong (since actions can overlap between states).